# 🔴 Interview: Paged Attention

vLLM-style paged KV cache — block table maps logical blocks to physical pages

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def paged_attention(Q, k_pages, v_pages, block_table, context_len, block_size):
    """
    手写分页注意力 —— 面试版。

    面试考点:
    1. 分页的思想来自操作系统虚拟内存：逻辑块→物理页的映射
    2. gather + 截取得到实际 KV 序列
    3. 手写 softmax: exp(scores - max) / sum(exp(...))

    参数:
        Q: (B, 1, D), k_pages/v_pages: (num_pages, block_size, D)
        block_table: (B, max_blocks), context_len: int, block_size: int
    返回: (B, 1, D)
    """
    B, _, D = Q.shape
    outputs = []
    scale = D ** -0.5

    for b in range(B):
        # ---- Step 1: Gather 物理页 ----
        phys = block_table[b]  # (max_blocks,)
        K = k_pages[phys].reshape(-1, D)[:context_len]  # (ctx, D)
        V = v_pages[phys].reshape(-1, D)[:context_len]  # (ctx, D)

        # ---- Step 2: 注意力分数 ----
        # Q[b:b+1]: (1, 1, D), K: (ctx, D)
        # scores: (1, 1, ctx)
        scores = (Q[b:b+1] @ K.unsqueeze(0).transpose(-2, -1)) * scale

        # ---- Step 3: 手写 softmax ----
        max_val = scores.max(dim=-1, keepdim=True).values  # (1, 1, 1)
        exp_s = torch.exp(scores - max_val)                 # (1, 1, ctx)
        attn = exp_s / exp_s.sum(dim=-1, keepdim=True)      # (1, 1, ctx)

        # ---- Step 4: 加权求和 ----
        out = attn @ V.unsqueeze(0)  # (1, 1, ctx) @ (1, ctx, D) = (1, 1, D)
        outputs.append(out)

    return torch.cat(outputs, dim=0)  # (B, 1, D)

In [ ]:
# Verify: paged output matches standard attention with contiguous layout
torch.manual_seed(0)
B, S, D = 2, 8, 16
block_size = 4
num_blocks = S // block_size

# Build contiguous K, V
K_full = torch.randn(B, S, D)
V_full = torch.randn(B, S, D)
Q = torch.randn(B, 1, D)

# Standard attention reference
scale = D ** -0.5
scores_ref = (Q @ K_full.transpose(-2, -1)) * scale
ref_out = torch.softmax(scores_ref, dim=-1) @ V_full

# Build paged structures: identity block table (pages in order)
total_pages = B * num_blocks
k_pages = torch.zeros(total_pages, block_size, D)
v_pages = torch.zeros(total_pages, block_size, D)
block_table = []
for b in range(B):
    page_ids = []
    for blk in range(num_blocks):
        pid = b * num_blocks + blk
        k_pages[pid] = K_full[b, blk*block_size:(blk+1)*block_size]
        v_pages[pid] = V_full[b, blk*block_size:(blk+1)*block_size]
        page_ids.append(pid)
    block_table.append(page_ids)

paged_out = paged_attention(Q, k_pages, v_pages, block_table, context_len=S, block_size=block_size)

print('Shape:', paged_out.shape)
print('Max diff vs reference:', (paged_out - ref_out).abs().max().item())
print('Match:', torch.allclose(paged_out, ref_out, atol=1e-5))

In [ ]:
from torch_judge import check
check('paged_attention')

## 📝 核心思路总结

1. **分页思想来自 OS**：KV 缓存按固定大小的「页」存储，block_table 做逻辑→物理映射，避免连续内存分配的碎片问题。
2. **Gather + 截取**：先用 block_table 高级索引取出物理页，reshape 拼接，再按 context_len 截取有效部分。
3. **逐 batch 处理**：每条序列的 block_table 不同，需要循环处理；实际 vLLM 会用更高效的批量 gather kernel。
4. **注意力计算不变**：分页只影响 KV 的存储和读取方式，注意力的计算逻辑（缩放点积 + softmax + 加权求和）与标准注意力完全一致。